<a href="https://colab.research.google.com/github/sidak1701/DataBricks-Hackathon/blob/main/LLM_as_RecSys_VLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "transformers<4.54.0"
!pip install vllm==0.9.1

In [ ]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from typing import Literal, List
from pydantic import BaseModel
import json

# Initialize vLLM with your chosen model
model_path = "Qwen/Qwen3-4B-Instruct-2507" # any model from HF, though qwen seem to work better with constrained generation
llm = LLM(model=model_path,
          dtype='half',
          max_model_len=1500 # model input context size
          )

INFO 10-29 21:57:16 [__init__.py:244] Automatically detected platform cuda.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 10-29 21:57:52 [config.py:823] This model supports multiple tasks: {'generate', 'embed', 'classify', 'score', 'reward'}. Defaulting to 'generate'.
WARNING 10-29 21:57:52 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 10-29 21:57:52 [arg_utils.py:1642] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 10-29 21:57:52 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1500, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fa

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 10-29 21:58:41 [default_loader.py:272] Loading weights took 44.55 seconds
INFO 10-29 21:58:42 [model_runner.py:1203] Model loading took 7.6065 GiB and 45.202076 seconds
INFO 10-29 21:58:44 [worker.py:294] Memory profiling takes 2.03 seconds
INFO 10-29 21:58:44 [worker.py:294] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.90) = 13.27GiB
INFO 10-29 21:58:44 [worker.py:294] model weights take 7.61GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 4.22GiB.
INFO 10-29 21:58:45 [executor_base.py:113] # cuda blocks: 1919, # CPU blocks: 1820
INFO 10-29 21:58:45 [executor_base.py:118] Maximum concurrency for 1500 tokens per request: 20.47x
INFO 10-29 21:58:48 [model_runner.py:1513] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in 

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 10-29 21:59:31 [model_runner.py:1671] Graph capturing finished in 43 secs, took 0.24 GiB
INFO 10-29 21:59:31 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 49.12 seconds


### Retireval test

we ask the model to retrieve relevant items from a list of different documents

In [ ]:
# Allowed document names (e.g., article titles in your library)
allowed_items = [
    "How to Fix a Leaky Faucet",
    "Harry Potter and the Sorcerer's Stone",
    "Cooking with Tofu",
    "How to Bake Croissants",
    "The Lord of the Rings",
    "Making Pastries at Home",
    "Inception",
    "Beginner’s Guide to Sourdough Bread",
    "Chocolate Cake Recipes",
    "Car Maintenance for Beginners",
    "History of Quantum Physics",
    "Batman The Dark Knight",
    "The Hobbit"
]

# Define the constrained generation schema
Items = Literal[tuple(allowed_items)]
class RetrievalResult(BaseModel):
    Items: List[Items]

schema = RetrievalResult.model_json_schema()

# Setup Guided Decoding parameters
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=150,
    guided_decoding=GuidedDecodingParams(json=schema),
    repetition_penalty=2, # optional: to prevent repeating a certain item
)

In [ ]:
prompt = "Retrive 3 different fantasy adventure movies."
result = llm.generate([prompt], sampling_params=sampling_params)
answer = result[0].outputs[0].text.strip()
articles = set(json.loads(answer)['Items'])
print("Items:", articles)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Items: {'The Lord of the Rings', "Harry Potter and the Sorcerer's Stone", 'The Hobbit'}


### Retrieval and Recommendation

In [ ]:
from enum import Enum

# an example with a imaginary user profile to select profiles better
user_profile = {
    "watch_history": ["Batman The Dark Knight",
                    "Harry Potter and the Sorcerer's Stone",
                    "Avengers End Game",
                    "The lord of the rings",
                    "The Inception",
                      "John Wick"
                      ],
}

prompt_with_profile = f"""Given the user profile:
Watch History: {', '.join(user_profile['watch_history'])}

Instructions:
- Recommend 1-3 movies.
- Do NOT repeat movies.
- Every suggestion must match user
- output persona and keep to one sentene
- output reason for each movie and keep reason to one sentence
"""

print(f"Prompt: \n---\n{prompt_with_profile}")


# Define the constrained generation schema
# Items = Literal[tuple(allowed_items)]

class Items(str, Enum):
    SpiderMan = "Spider-man: far from home"
    Hobbit = "The Hobbit"
    Starwars = "Star Wars"
    DeadPool = "Dead Pool"
    Avengers = "Avengers Age of Ultron"
    HarryPotter = "Harry Potter and the Goblet of Fire"
    Samurai ="The last samurai"
    PrideAndPrejudice = "Pride and Prejudice"
    WhenHarryMetSally = "When Harry Met Sally..."
    Bridesmaids = "Bridesmaids"
    EmilyInParis = "Emily In Paris"
    Shrek = "Shrek"
    TenThingsIHate = "10 Things I Hate About You"
    MeanGirls = "Mean Girls"
    Tangled = "Tangled"
    HowIMetYourMother = "How I met your mother"

class ItemDesc(BaseModel):
    title: Items
    reason: str

class RetrievalResult(BaseModel):
    Persona: str
    Movies: List[ItemDesc]

schema = RetrievalResult.model_json_schema()

# Setup Guided Decoding parameters
sampling_params = SamplingParams(
    temperature=0.1,
    max_tokens=1500,
    guided_decoding=GuidedDecodingParams(json=schema),
    # repetition_penalty=1, # optional: to prevent repeating a certain item
)

result_with_profile = llm.generate([prompt_with_profile], sampling_params=sampling_params)
try:
    answer_with_profile = json.loads(result_with_profile[0].outputs[0].text.strip())
except:
    answer_with_profile = result_with_profile[0].outputs[0].text.strip()

import json
from pprint import pprint

print("Answer:\n---\n")
pprint(answer_with_profile)

Prompt: 
---
Given the user profile:
Watch History: Batman The Dark Knight, Harry Potter and the Sorcerer's Stone, Avengers End Game, The lord of the rings, The Inception, John Wick

Instructions:
- Recommend 1-3 movies.
- Do NOT repeat movies.
- Every suggestion must match user
- output persona and keep to one sentene
- output reason for each movie and keep reason to one sentence



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Answer:
---

{'Movies': [{'reason': 'A blend of action and dark humor with a complex '
                       "narrative structure that mirrors the user's taste for "
                       'intricate plots and high-octane sequences.',
             'title': 'Dead Pool'},
            {'reason': 'A visually stunning action film with a rich, layered '
                       'narrative that combines historical depth with intense '
                       "combat sequences, appealing to the user's love for "
                       'dramatic and complex storytelling.',
             'title': 'The last samurai'},
            {'reason': 'A visually stunning action film with a rich, layered '
                       'narrative that combines historical depth with intense '
                       "combat sequences, appealing to the user's love for "
                       'dramatic and complex storytelling.',
             'title': 'The last samurai'}],
 'Persona': 'A seasoned action and adventure en